# Prompting Large Language Models

In this notebook we explore how to interact with a Large Language Model (LLM) through **prompt engineering**. Effective prompting can often replace fine-tuning for many tasks — saving significant compute.

We use the same model, the same `text-generation` pipeline and the same `generate()` helper as the *Basics of Text Generation* notebook, so everything you learned there about decoding applies here unchanged.

We cover:
1. Loading an instruction-tuned model as a `text-generation` pipeline
2. Chat templates and the `generate()` helper
3. Zero-shot prompting
4. Few-shot prompting
5. System prompts
6. Chain-of-Thought (CoT) prompting
7. Structured output

## Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `transformers` | The `pipeline` API and all generation parameters |
| `accelerate` | Device placement (`device_map="auto"`) and efficient loading |

No `bitsandbytes`: this notebook runs the model **unquantised**. A 1.5 B-parameter model in `bfloat16` needs roughly 3 GB of GPU memory, which fits comfortably on a free Colab T4.

In [1]:
#!pip install -q -U transformers accelerate

In [2]:
from google.colab import output
output.enable_custom_widget_manager()

In [3]:
import json
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

from transformers import pipeline
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

## Loading the Model with `pipeline()`

We use **Qwen/Qwen2.5-1.5B-Instruct** — a compact (1.5 B parameters), instruction-tuned model that runs comfortably on a single GPU. It is the same model as in the *Basics of Text Generation* notebook.

`pipeline("text-generation", model=...)` does four things for us in one line:

1. downloads and instantiates the model,
2. downloads and instantiates the matching tokenizer,
3. places the weights on the available device (`device_map="auto"`),
4. wraps tokenise → `generate()` → decode into a single callable.

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

pipe = pipeline(
    task="text-generation",
    model=MODEL_NAME,
    dtype="auto",        # keep the checkpoint's native precision (bfloat16 for Qwen2.5)
    device_map="auto",   # put the weights on the GPU if there is one
)

print(f"Model:  {MODEL_NAME}")
print(f"Device: {pipe.model.device}")
print(f"Dtype:  {pipe.model.dtype}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model:  Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Dtype:  torch.bfloat16


## Chat Templates and the `generate()` Helper

Modern instruction-tuned models expect input in a specific structured format called a **chat template**. Using the wrong format degrades performance, because the model was fine-tuned to expect a particular delimiter pattern.

The `generate()` helper below is the one from the *Basics of Text Generation* notebook. It builds the chat from a prompt and an optional `system=` message, and passes every other keyword argument straight through to the pipeline. The pipeline returns the whole conversation with the reply appended as the last message, which is why the helper ends in `[-1]["content"]`.

In [5]:
def generate(prompt, system="You are a helpful assistant.", **kwargs):
    """One completion. Extra keyword arguments are forwarded to generate()."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    result = pipe(messages, **kwargs)
    return result[0]["generated_text"][-1]["content"]


print(generate("Say hello in one short sentence.", do_sample=False, max_new_tokens=30))

Hello! How can I assist you today?


## Prompting Strategies

The sections below cover the main prompting techniques from simplest to most sophisticated.

### 1. Zero-Shot Prompting

Ask the model to perform a task with **no examples** — relying entirely on pre-trained knowledge.

**When to use:** The task is common and well-represented in the training data (translation, summarisation, simple classification).

**Limitations:** Can fail on domain-specific or unusual tasks.

In [6]:
response = generate(
    "Classify the sentiment of this text as positive, neutral, or negative:\n"
    "'I think the vacation was okay.'",
    do_sample=False, max_new_tokens=100,
)
print(response)

The sentiment of the text "I think the vacation was okay." is neutral.


### 2. Few-Shot Prompting

Provide a handful of labelled examples (**shots**) in the prompt before the actual query. The model performs **in-context learning** — it infers the task format and label space from the examples without any weight updates.

**Key insight:** The examples do not fine-tune the model; they influence only the attention pattern for that single forward pass.

**When to use:** The task is novel, or the zero-shot output format is inconsistent.

In [7]:
few_shot_prompt = """Classify each review as positive, neutral, or negative.

Review: 'This restaurant is the best I've ever been to. Delicious food, friendly staff.'
Sentiment: positive

Review: 'I was disappointed — the product broke within a week.'
Sentiment: negative

Review: 'The movie was okay, not great but not terrible either.'
Sentiment: neutral

Review: 'I absolutely love the new Spider-Man film. Incredibly well done!'
Sentiment:"""

response = generate(few_shot_prompt, do_sample=False, max_new_tokens=100)
print(response)

positive


### 3. System Prompts

The `system` role in a chat template lets you define the model's **persona, rules, and constraints** that persist for the whole conversation. Unlike user instructions, system prompts are typically hidden from end users.

**Use cases:** Chatbots with specific personalities, content-filtered assistants, domain-constrained tools.

In [8]:
system = (
    "You are a grumpy pirate who answers every question correctly "
    "but complains about having to do so. Always end with 'Arrr!'"
)

response = generate("What is the capital of France?", system=system, do_sample=False, max_new_tokens=150)
print(response)

The capital of France is Paris, and it's where I sail my ship! Arrr!


### 4. Chain-of-Thought (CoT) Prompting

For **reasoning-intensive tasks** (arithmetic, logic, multi-step problems), LLMs perform significantly better when instructed to *reason step by step* before giving the final answer (Wei et al., 2022).

The key insight: generating intermediate reasoning steps provides additional context tokens that guide the model toward the correct answer — essentially using the output space as a scratchpad.

**Variants:**
- **Zero-shot CoT:** Append *"Let's think step by step."* to the prompt.
- **Few-shot CoT:** Include worked examples that show reasoning steps.
- **Self-consistency:** Sample multiple CoT paths and take a majority vote.

In [9]:
problem = (
    "A store sells apples for €1.20 each. A customer buys 3 apples and pays with a €5 note. "
    "How much change does the customer receive?"
)

print("=== Without CoT ===")
print(generate(problem, do_sample=False, max_new_tokens=300))

print("\n=== With Zero-Shot CoT ===")
print(generate(problem + "\n\nLet's think step by step.", do_sample=False, max_new_tokens=300))

=== Without CoT ===
To determine how much change the customer receives, we need to follow these steps:

1. Calculate the total cost of the apples.
2. Subtract the total cost from the amount paid.

First, let's calculate the total cost of the apples. The store sells apples for €1.20 each, and the customer buys 3 apples. So, the total cost is:
\[ 3 \text{ apples} \times €1.20 \text{ per apple} = €3.60 \]

Next, we subtract the total cost from the amount paid by the customer. The customer pays with a €5 note. So, the change received is:
\[ €5.00 - €3.60 = €1.40 \]

Therefore, the change the customer receives is \(\boxed{1.40}\).

=== With Zero-Shot CoT ===
Sure, let's break it down step by step:

1. **Determine the cost of one apple:**
   The store sells apples for €1.20 each.

2. **Calculate the total cost for 3 apples:**
   \[
   \text{Total cost} = \text{Cost per apple} \times \text{Number of apples}
   \]
   \[
   \text{Total cost} = €1.20 \times 3 = €3.60
   \]

3. **Subtract the tot

### 5. Structured Output

Every technique so far returned prose. A research pipeline needs a **row**: a label, perhaps a flag, perhaps the span of text the label came from — in a shape that goes into a dataframe without hand-cleaning. Prose that needs a human to read it before it can be counted is not an analysis pipeline.

Three things make that reliable, in increasing order of strength:

1. **State the schema.** Name the exact keys and the allowed values in the prompt.
2. **Show one filled example.** Few-shot prompting, applied to the format rather than to the task.
3. **Prefill the answer.** End the chat with an `assistant` message that already contains the opening of the JSON. The pipeline *continues* that message instead of starting a new one (as in section 14 of the *Basics* notebook), so the model cannot preface its answer with "Sure! Here is the JSON:" — it is already inside the object.

Prefill is what turns a usually-parses prompt into an almost-always-parses one. Note what it does not do: the model can still emit a value outside your scheme, or stop before closing a brace. Parse defensively, and count the failures — they are missing data.

In [10]:
SYSTEM = "You label public comments. Reply with JSON only."


def schema_prompt(comment):
    return (
        "Code this public comment about a proposed wind farm.\n\n"
        "Return JSON with exactly these keys:\n"
        '  "stance"   : one of "support", "oppose", "unclear"\n'
        '  "condition": true if the stance depends on a condition being met, else false\n'
        '  "evidence" : the shortest quote from the comment that justifies the stance\n\n'
        f"Comment: {comment}"
    )


COMMENT = "If the access road is rebuilt and noise limits are respected, I would back it."

print("=== schema only ===")
print(generate(schema_prompt(COMMENT), system=SYSTEM, do_sample=False, max_new_tokens=120))

print("\n=== schema + assistant prefill ===")
messages = [
    {"role": "system",    "content": SYSTEM},
    {"role": "user",      "content": schema_prompt(COMMENT)},
    {"role": "assistant", "content": '{"stance": "'},   # the model continues THIS message
]
print(pipe(messages, max_new_tokens=120, do_sample=False)[0]["generated_text"][-1]["content"])

=== schema only ===
```json
{
  "stance": "support",
  "condition": true,
  "evidence": "If the access road is rebuilt and noise limits are respected, I would back it."
}
```

=== schema + assistant prefill ===
{"stance": "support", "condition": true, "evidence": "If the access road is rebuilt and noise limits are respected"}


In [11]:
import pandas as pd
import json

COMMENTS = [
    "I fully support the project. We need the energy and the jobs it would bring.",
    "This will destroy the landscape my family has farmed for four generations. No.",
    "If the access road is rebuilt and noise limits are respected, I would back it.",
    "Nobody from the council has answered a single one of my letters since March.",
]


def code_comment(comment):
    """One comment in, one parsed row out. An unparseable reply becomes an empty row."""
    messages = [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": schema_prompt(comment)},
        {"role": "assistant", "content": '{"stance": "'},
    ]
    reply = pipe(messages, max_new_tokens=120, do_sample=False)[0]["generated_text"][-1]["content"]
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        return {"stance": None, "condition": None, "evidence": None}   # missing, and countable


pd.DataFrame([code_comment(c) for c in COMMENTS], index=[c[:45] + "…" for c in COMMENTS])

,stance,condition,evidence
I fully support the project. We need the ener…,support,False,We need the energy
This will destroy the landscape my family has…,oppose,False,This will destroy the landscape my family has ...
If the access road is rebuilt and noise limit…,support,True,If the access road is rebuilt and noise limits...
Nobody from the council has answered a single…,oppose,False,Nobody from the council has answered a single ...


### Summary: When to Use Which Prompting Strategy

| Strategy | Best for | Limitations |
|---|---|---|
| Zero-shot | Common, well-defined tasks | May fail on unusual tasks |
| Few-shot | Novel tasks needing format guidance | Uses context window space |
| System prompt | Persona / constraint setting | Model may not always comply |
| Chain-of-Thought | Multi-step reasoning, math | Longer outputs; slower; breaks simple parsers |
| Structured output | Anything whose result becomes data | Still needs defensive parsing |

**Beyond this notebook.** *ReAct* (Yao et al., 2022) interleaves reasoning with tool calls — Thought, Action, Observation, repeat — and is the pattern underneath most agent frameworks. It matters when the model needs to fetch or compute something mid-answer, which is a different problem from the ones here.

For more techniques: https://www.promptingguide.ai/techniques

## References

* HuggingFace pipelines — https://huggingface.co/docs/transformers/main_classes/pipelines
* Chat templates — https://huggingface.co/docs/transformers/chat_templating
* Chain-of-Thought prompting (Wei et al., 2022) — https://arxiv.org/abs/2201.11903
* Self-consistency (Wang et al., 2023) — https://arxiv.org/abs/2203.11171
* ReAct (Yao et al., 2022) — https://arxiv.org/abs/2210.03629
* Prompting guide — https://www.promptingguide.ai/
* Qwen2.5 model — https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct